In [1]:
import pandas as pd
import numpy as np

# 1. Muat dataset
file_path = "data/ev battery_failure prediction Dataset.csv"
df = pd.read_csv(file_path)

print(f"Total baris: {df.shape[0]:,}")
print(f"Total kolom: {df.shape[1]}")

# 2. Periksa distribusi kelas target
target_col = "battery_failure"
target_counts = df[target_col].value_counts()
target_pct = df[target_col].value_counts(normalize=True) * 100

print("\n--- Distribusi Target (battery_failure) ---")
for val in target_counts.index:
    print(f"Kelas {val}: {target_counts[val]:,} baris ({target_pct[val]:.2f}%)")

# 3. Cek missing values dan ringkasan tipe data
null_total = df.isnull().sum().sum()
print(f"\nTotal missing values di dataset: {null_total}")
print("\nSebaran tipe data:")
print(df.dtypes.value_counts())

# 4. Cetak seluruh daftar 70 nama kolom
print("\n--- Daftar Lengkap Kolom ---")
for idx, col in enumerate(df.columns, 1):
    print(f"{idx}. {col} ({df[col].dtype})")

Total baris: 20,000
Total kolom: 70

--- Distribusi Target (battery_failure) ---
Kelas 0: 18,616 baris (93.08%)
Kelas 1: 1,384 baris (6.92%)

Total missing values di dataset: 53344

Sebaran tipe data:
float64    59
str        10
int64       1
Name: count, dtype: int64

--- Daftar Lengkap Kolom ---
1. vehicle_id (str)
2. vehicle_brand (str)
3. vehicle_model (str)
4. vehicle_type (str)
5. manufacturing_year (float64)
6. battery_manufacturer (str)
7. battery_chemistry (str)
8. battery_capacity_kwh (float64)
9. drive_type (str)
10. odometer_km (float64)
11. vehicle_age_years (float64)
12. fleet_or_private (str)
13. battery_serial (str)
14. cycle_count (float64)
15. battery_health_percent (float64)
16. state_of_charge (float64)
17. depth_of_discharge (float64)
18. state_of_health (float64)
19. cell_voltage_avg (float64)
20. cell_voltage_std (float64)
21. pack_voltage (float64)
22. cell_temperature_avg (float64)
23. cell_temperature_max (float64)
24. internal_resistance (float64)
25. charge_

In [2]:
# 1. Analisis Missing Values (Kolom dengan missing values terbanyak)
null_series = df.isnull().sum()
null_pct = (null_series / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': null_series, 
    'Percentage': null_pct
}).query('Missing_Count > 0').sort_values(by='Missing_Count', ascending=False)

print(f"Jumlah kolom yang memiliki missing values: {len(missing_df)} dari 70 kolom")
print("\nTop 15 Kolom dengan Missing Values Terbanyak:")
print(missing_df.head(15))

# 2. Cek variasi nilai unik pada 10 kolom string/kategorial
print("\n--- Ringkasan Kolom Kategorial (String) ---")
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_cols:
    n_unique = df[col].nunique()
    samples = df[col].dropna().unique()[:4]
    print(f"• {col:<22} | Unik: {n_unique:<5} | Contoh nilai: {list(samples)}")

# 3. Tampilkan seluruh 70 nama kolom ke dalam list bersih
print("\n--- Semua 70 Kolom (Dikelompokkan per baris) ---")
all_columns = df.columns.tolist()
for i in range(0, len(all_columns), 5):
    print(f"{i+1:02d}-{min(i+5, len(all_columns)):02d}: {', '.join(all_columns[i:i+5])}")

Jumlah kolom yang memiliki missing values: 67 dari 70 kolom

Top 15 Kolom dengan Missing Values Terbanyak:
                        Missing_Count  Percentage
fast_charge_ratio                 988        4.94
average_speed                     984        4.92
manufacturing_year                982        4.91
home_charging_ratio               980        4.90
sensor_fault_count                968        4.84
charging_interruptions            964        4.82
state_of_charge                   964        4.82
state_of_health                   956        4.78
capacity_loss_percent             956        4.78
slow_charge_ratio                 948        4.74
cell_temperature_max              944        4.72
cooling_system_health             940        4.70
cell_voltage_avg                  940        4.70
charge_efficiency                 930        4.65
hard_braking_score                924        4.62

--- Ringkasan Kolom Kategorial (String) ---
• vehicle_id             | Unik: 20000 | Contoh 

In [3]:
# 1. Hapus identifier yang tidak memiliki nilai prediktif
df_clean = df.drop(columns=['vehicle_id'])

# 2. Ambil semua fitur numerik dan hitung korelasi dengan target
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
correlations = df_clean[numeric_cols].corr()['battery_failure'].sort_values(ascending=False)

print("--- Top 10 Fitur dengan Korelasi Positif Tertinggi ke battery_failure ---")
print(correlations.head(11).iloc[1:]) # Lewati target itu sendiri

print("\n--- Top 10 Fitur dengan Korelasi Negatif Terkuat ke battery_failure ---")
print(correlations.tail(10))

# 3. Cek apakah ada fitur yang memiliki korelasi ekstrem (|corr| > 0.85) -> Potensi Leakage
suspicious_features = correlations[abs(correlations) > 0.85].index.tolist()
suspicious_features = [col for col in suspicious_features if col != 'battery_failure']

print(f"\nFitur mencurigakan (indikasi data leakage |corr| > 0.85): {len(suspicious_features)}")
if suspicious_features:
    print("Kolom terdeteksi:", suspicious_features)
else:
    print("Tidak ada fitur leakage tunggal yang mendominasi secara ekstrem.")

--- Top 10 Fitur dengan Korelasi Positif Tertinggi ke battery_failure ---
thermal_runaway_risk     0.390162
capacity_loss_percent    0.313285
internal_resistance      0.289066
cell_temperature_max     0.286755
cell_temperature_avg     0.265972
battery_stress_index     0.239195
aging_score              0.229643
vehicle_age_years        0.219600
previous_faults          0.194726
BMS_warning_count        0.163067
Name: battery_failure, dtype: float64

--- Top 10 Fitur dengan Korelasi Negatif Terkuat ke battery_failure ---
maintenance_score                 -0.040416
cooling_system_health             -0.090674
remaining_capacity                -0.091528
manufacturing_year                -0.216856
discharge_efficiency              -0.251558
charge_efficiency                 -0.259309
predicted_remaining_life_cycles   -0.300278
state_of_health                   -0.309538
battery_health_percent            -0.312144
thermal_health_score              -0.382988
Name: battery_failure, dtype: float

In [6]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, 
    average_precision_score, 
    classification_report, 
    confusion_matrix
)
import matplotlib.pyplot as plt

# 1. Siapkan fitur (X) dan target (y)
drop_cols = ['battery_failure']
if 'vehicle_id' in df.columns:
    drop_cols.append('vehicle_id')

X = df.drop(columns=drop_cols).copy()
y = df['battery_failure'].copy()

# 2. Konversi kolom objek/string menjadi tipe category untuk LightGBM
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].astype('category')

# 3. Hitung bobot penyeimbang kelas
scale_weight = (len(y) - sum(y)) / sum(y) # ~13.45

# 4. Inisialisasi Stratified 5-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_preds = np.zeros(len(df))
feature_importances = pd.DataFrame(index=X.columns)

roc_scores = []
pr_auc_scores = []

print("Memulai Training Stratified 5-Fold LightGBM...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(
        n_estimators=600,
        learning_rate=0.03,
        scale_pos_weight=scale_weight,
        random_state=42,
        importance_type='gain',
        verbose=-1
    )
    
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    # Prediksi probabilitas kelas 1 (gagal)
    val_probs = model.predict_proba(X_val)[:, 1]
    oof_preds[val_idx] = val_probs
    
    # Evaluasi metrik fold
    fold_roc = roc_auc_score(y_val, val_probs)
    fold_pr_auc = average_precision_score(y_val, val_probs)
    roc_scores.append(fold_roc)
    pr_auc_scores.append(fold_pr_auc)
    
    feature_importances[f'fold_{fold}'] = model.feature_importances_
    print(f"Fold {fold} | ROC-AUC: {fold_roc:.4f} | PR-AUC: {fold_pr_auc:.4f}")

# 5. Rangkuman Metrik Validasi
mean_roc = np.mean(roc_scores)
mean_pr_auc = np.mean(pr_auc_scores)
oof_binary = (oof_preds >= 0.5).astype(int)

print("\n" + "="*45)
print(f"Rata-rata ROC-AUC : {mean_roc:.4f}")
print(f"Rata-rata PR-AUC  : {mean_pr_auc:.4f}")
print("="*45)
print("\nClassification Report (Threshold 0.5):")
print(classification_report(y, oof_binary, digits=4))

# 6. Top 15 Fitur Paling Berpengaruh (Gain Importance)
feature_importances['mean_importance'] = feature_importances.mean(axis=1)
top_features = feature_importances['mean_importance'].sort_values(ascending=False).head(15)

print("\nTop 15 Fitur Paling Berpengaruh (Berdasarkan Gain Split):")
print(top_features)

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\lightgbm\basic.py:30: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.2)
  import scipy.sparse


Memulai Training Stratified 5-Fold LightGBM...



C:\Users\User\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 | ROC-AUC: 0.9824 | PR-AUC: 0.7815


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

Fold 3 | ROC-AUC: 0.9828 | PR-AUC: 0.7515


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4 | ROC-AUC: 0.9848 | PR-AUC: 0.7828


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 | ROC-AUC: 0.9846 | PR-AUC: 0.7784

Rata-rata ROC-AUC : 0.9833
Rata-rata PR-AUC  : 0.7664

Classification Report (Threshold 0.5):
              precision    recall  f1-score   support

           0     0.9867    0.9744    0.9805     18616
           1     0.7050    0.8237    0.7597      1384

    accuracy                         0.9639     20000
   macro avg     0.8459    0.8990    0.8701     20000
weighted avg     0.9672    0.9639    0.9652     20000


Top 15 Fitur Paling Berpengaruh (Berdasarkan Gain Split):
thermal_runaway_risk               117305.378769
battery_health_percent              77622.289386
previous_faults                     69434.065630
thermal_health_score                69007.861877
BMS_warning_count                   38822.465316
capacity_loss_percent               38749.018673
state_of_health                     28382.156687
vehicle_model                       21021.769621
predicted_remaining_life_cycles     19740.492097
abnormal_voltage_events             

In [7]:
# Filter fitur yang berkaitan dengan kebiasaan/operasional pengemudi
actionable_keywords = ['charge', 'charging', 'speed', 'braking', 'soc', 'state_of_charge', 'depth']

actionable_mask = feature_importances.index.str.lower().map(
    lambda col: any(kw in col for kw in actionable_keywords)
)

actionable_importance = feature_importances.loc[actionable_mask, 'mean_importance'].sort_values(ascending=False)

print("--- Peringkat Fitur Kebiasaan Pengemudi (Actionable) ---")
print(actionable_importance)

--- Peringkat Fitur Kebiasaan Pengemudi (Actionable) ---
discharge_efficiency          5155.107715
charge_efficiency             3227.137169
hard_braking_score            2254.439521
state_of_charge               1863.758440
depth_of_discharge            1779.839890
average_charging_time         1561.160410
average_speed                 1545.842329
regenerative_braking_usage    1521.544048
average_charge_power_kw       1385.759048
charging_quality_score        1364.767225
slow_charge_ratio             1345.711526
overnight_charging_ratio      1293.502659
home_charging_ratio           1237.439177
fast_charge_ratio             1082.700049
charging_cycles_last_month     951.719753
charging_interruptions         188.744913
overcharge_events              125.876450
Name: mean_importance, dtype: float64


In [8]:
import joblib
import time
from scipy.optimize import minimize

# 1. Latih model final pada seluruh data dan simpan
final_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    scale_pos_weight=scale_weight,
    random_state=42,
    verbose=-1
)
final_model.fit(X, y)

# Simpan model untuk API nanti
joblib.dump(final_model, 'ev_battery_lgbm.joblib')
print("Model berhasil disimpan ke: ev_battery_lgbm.joblib")

# 2. Ambil 1 sampel mobil dengan risiko kegagalan tinggi
high_risk_idx = np.where((y == 1) & (oof_preds > 0.80))[0][0]
sample_car = X.iloc[[high_risk_idx]].copy()
initial_risk = final_model.predict_proba(sample_car)[0, 1]

print(f"\n--- Sampel Mobil Uji (Indeks {high_risk_idx}) ---")
print(f"Probabilitas Kerusakan Awal: {initial_risk * 100:.2f}% (RISIKO TINGGI)")

# 3. Definisikan 5 Fitur Actionable & Batas Nilai Realistis (Bounds)
actionable_cols = [
    'depth_of_discharge',   # Rentang wajar: 30% - 90%
    'state_of_charge',      # Rentang wajar: 50% - 100%
    'fast_charge_ratio',    # Rentang wajar: 0.0 - 1.0
    'hard_braking_score',   # Rentang skor berkendara: 0 - 100
    'average_speed'         # Kecepatan rata-rata (km/jam): 20 - 120
]

bounds = [
    (30.0, 85.0),   # depth_of_discharge
    (60.0, 95.0),   # state_of_charge
    (0.05, 0.50),   # fast_charge_ratio (anjuran kurangi fast charge)
    (0.0, 50.0),    # hard_braking_score (perbaiki gaya rem)
    (30.0, 90.0)    # average_speed
]

# Ambil nilai awal mobil untuk 5 fitur tersebut
x0 = sample_car[actionable_cols].values[0].astype(float)
stds = X[actionable_cols].std().values # Skala normalisasi jarak perubahan

# 4. Fungsi Optimizer What-If Terarah
def optimize_recommendation(car_row, target_risk_threshold=0.20):
    start_time = time.perf_counter()
    temp_row = car_row.copy()
    
    # Fungsi objektif: Minimalkan deviasi perubahan dari kebiasaan awal pengemudi
    def loss(x_act):
        penalty_distance = np.sum(((x_act - x0) / stds) ** 2)
        
        # Evaluasi risiko baru dengan nilai usulan
        for i, col in enumerate(actionable_cols):
            temp_row[col] = x_act[i]
        risk = final_model.predict_proba(temp_row)[0, 1]
        
        # Penalti berat jika risiko masih di atas threshold aman (20%)
        risk_penalty = 50.0 * max(0.0, risk - target_risk_threshold) ** 2
        return penalty_distance + risk_penalty

    # Eksekusi pencarian solusi optimal via L-BFGS-B
    res = minimize(loss, x0, method='L-BFGS-B', bounds=bounds, options={'maxiter': 35})
    
    # Hitung risiko final
    for i, col in enumerate(actionable_cols):
        temp_row[col] = res.x[i]
    new_risk = final_model.predict_proba(temp_row)[0, 1]
    
    elapsed_ms = (time.perf_counter() - start_time) * 1000
    return res.x, new_risk, elapsed_ms

# 5. Uji Kecepatan dan Efektivitas Rekomendasi
optimal_values, reduced_risk, exec_time = optimize_recommendation(sample_car)

print(f"Komputasi selesai dalam: {exec_time:.2f} ms")
print(f"Probabilitas Kerusakan Baru: {reduced_risk * 100:.2f}%\n")

print("--- REKOMENDASI PERUBAHAN KEBIASAAN UNTUK PENGEMUDI ---")
for col, old_val, new_val in zip(actionable_cols, x0, optimal_values):
    delta = new_val - old_val
    arah = "Turunkan" if delta < 0 else "Naikkan"
    print(f"• {col:<22}: {old_val:6.1f}  ──>  {new_val:6.1f} ({arah} {abs(delta):.1f})")

Model berhasil disimpan ke: ev_battery_lgbm.joblib

--- Sampel Mobil Uji (Indeks 19) ---
Probabilitas Kerusakan Awal: 99.09% (RISIKO TINGGI)
Komputasi selesai dalam: 208.69 ms
Probabilitas Kerusakan Baru: 99.18%

--- REKOMENDASI PERUBAHAN KEBIASAAN UNTUK PENGEMUDI ---
• depth_of_discharge    :   59.0  ──>    59.0 (Naikkan 0.0)
• state_of_charge       :   20.3  ──>    60.0 (Naikkan 39.7)
• fast_charge_ratio     :    0.5  ──>     0.5 (Naikkan 0.0)
• hard_braking_score    :   30.9  ──>    30.9 (Naikkan 0.0)
• average_speed         :  114.0  ──>    90.0 (Turunkan 24.0)


In [9]:
# 1. Inspeksi kondisi fisik internal sampel 19
print("--- Diagnostik Fisik Mobil Indeks 19 ---")
health_cols = [
    'battery_health_percent', 'capacity_loss_percent', 
    'internal_resistance', 'thermal_runaway_risk', 'state_of_health'
]
print(sample_car[health_cols].T)

# 2. Fast Coordinate Search Optimizer (Khusus Tree-Based Models)
def fast_tree_counterfactual(car_row, target_risk=0.30, steps_per_feature=8):
    start_time = time.perf_counter()
    
    # Salin baris data
    best_row = car_row.copy()
    current_x = car_row[actionable_cols].values[0].astype(float)
    best_x = current_x.copy()
    
    # Hitung risiko awal
    best_risk = final_model.predict_proba(best_row)[0, 1]
    
    # Buat grid nilai diskrit untuk tiap fitur actionable
    feature_grids = {
        col: np.linspace(b[0], b[1], steps_per_feature)
        for col, b in zip(actionable_cols, bounds)
    }
    
    # Iterasi terarah: Uji perubahan nilai per fitur dalam batch
    for iteration in range(2):  # 2 putaran optimasi koordinat
        for i, col in enumerate(actionable_cols):
            candidate_vals = feature_grids[col]
            
            # Buat batch data untuk evaluasi instan sekaligus
            batch = pd.concat([best_row] * len(candidate_vals), ignore_index=True)
            batch[col] = candidate_vals
            
            # Prediksi serentak (vectorized prediction)
            probs = final_model.predict_proba(batch)[:, 1]
            
            # Hitung skor penalti gabungan (risiko + deviasi kebiasaan awal)
            distances = ((candidate_vals - current_x[i]) / stds[i]) ** 2
            scores = probs + (0.05 * distances)
            
            # Pilih nilai terbaik yang paling menekan risiko
            best_idx = np.argmin(scores)
            best_row[col] = candidate_vals[best_idx]
            best_x[i] = candidate_vals[best_idx]
            best_risk = probs[best_idx]

    exec_time = (time.perf_counter() - start_time) * 1000
    return best_x, best_risk, exec_time

# 3. Uji pada sampel mobil yang berada di ambang batas risiko akibat kebiasaan
# Cari sampel dengan risiko 65% - 85% (borderline risk)
borderline_indices = np.where((oof_preds > 0.65) & (oof_preds < 0.85))[0]
test_idx = borderline_indices[0] if len(borderline_indices) > 0 else high_risk_idx

test_car = X.iloc[[test_idx]].copy()
initial_p = final_model.predict_proba(test_car)[0, 1]

print(f"\n=======================================================")
print(f"Uji Sampel Indeks {test_idx} | Risiko Awal: {initial_p * 100:.2f}%")
print(f"=======================================================")

opt_values, final_p, duration = fast_tree_counterfactual(test_car)

print(f"Komputasi selesai dalam: {duration:.2f} ms")
print(f"Risiko Berhasil Diturunkan Menjadi: {final_p * 100:.2f}%\n")

print("--- REKOMENDASI PERUBAHAN KEBIASAAN ---")
for col, old_v, new_v in zip(actionable_cols, test_car[actionable_cols].values[0], opt_values):
    delta = new_v - old_v
    perubahan = "Pertahankan" if abs(delta) < 0.1 else ("Turunkan" if delta < 0 else "Naikkan")
    print(f"• {col:<22}: {old_v:6.1f}  ──>  {new_v:6.1f} ({perubahan} {abs(delta):.1f})")

--- Diagnostik Fisik Mobil Indeks 19 ---
                            19
battery_health_percent  69.490
capacity_loss_percent   30.510
internal_resistance      0.804
thermal_runaway_risk    51.160
state_of_health         70.470

Uji Sampel Indeks 15 | Risiko Awal: 21.35%
Komputasi selesai dalam: 317.34 ms
Risiko Berhasil Diturunkan Menjadi: 26.82%

--- REKOMENDASI PERUBAHAN KEBIASAAN ---
• depth_of_discharge    :   34.3  ──>    30.0 (Turunkan 4.3)
• state_of_charge       :    4.7  ──>    60.0 (Naikkan 55.3)
• fast_charge_ratio     :    0.4  ──>     0.4 (Pertahankan 0.0)
• hard_braking_score    :   89.9  ──>    50.0 (Turunkan 39.9)
• average_speed         :    nan  ──>    30.0 (Naikkan nan)


In [10]:
# 1. Hitung median untuk fallback jika ada nilai NaN pada input pengguna
actionable_medians = X[actionable_cols].median()

# 2. Cari sampel mobil berisiko tinggi yang fiturnya bersih (tanpa NaN pada actionable cols)
all_preds = final_model.predict_proba(X)[:, 1]
valid_mask = ~X[actionable_cols].isnull().any(axis=1) & (all_preds >= 0.60) & (all_preds <= 0.85)
valid_candidates = np.where(valid_mask)[0]

target_idx = valid_candidates[0]
sample_clean = X.iloc[[target_idx]].copy()
initial_risk_val = final_model.predict_proba(sample_clean)[0, 1]

# 3. Fast Optimizer yang Robust (Tahan NaN & Wajib Menurunkan Risiko)
def robust_tree_counterfactual(car_row, target_risk=0.25, steps=10):
    start_time = time.perf_counter()
    
    # Isi NaN pada fitur actionable dengan median jika ada
    clean_row = car_row.copy()
    for col in actionable_cols:
        if pd.isna(clean_row[col].values[0]):
            clean_row[col] = actionable_medians[col]
            
    best_row = clean_row.copy()
    current_x = clean_row[actionable_cols].values[0].astype(float)
    best_x = current_x.copy()
    best_risk = final_model.predict_proba(best_row)[0, 1]
    
    # Grid eksplorasi
    feature_grids = {
        col: np.linspace(b[0], b[1], steps)
        for col, b in zip(actionable_cols, bounds)
    }
    
    # Koordinat iteratif
    for _ in range(2):
        for i, col in enumerate(actionable_cols):
            candidate_vals = feature_grids[col]
            batch = pd.concat([best_row] * len(candidate_vals), ignore_index=True)
            batch[col] = candidate_vals
            
            probs = final_model.predict_proba(batch)[:, 1]
            
            # Cari nilai yang menurunkan risiko dan meminimalkan deviasi
            for cand_val, p in zip(candidate_vals, probs):
                if p < best_risk:  # Guardrail: Hanya terima jika risiko benar-benar turun
                    best_risk = p
                    best_x[i] = cand_val
                    best_row[col] = cand_val

    exec_time = (time.perf_counter() - start_time) * 1000
    return best_x, best_risk, exec_time

# 4. Eksekusi Simulasi
opt_vals, reduced_p, dur = robust_tree_counterfactual(sample_clean)

print("=" * 55)
print(f"Sampel Mobil Terpilih : Indeks {target_idx}")
print(f"Probabilitas Awal     : {initial_risk_val * 100:.2f}% (BERBAHAYA)")
print(f"Probabilitas Baru     : {reduced_p * 100:.2f}% (ZONA AMAN)")
print(f"Waktu Inferensi       : {dur:.2f} ms")
print("=" * 55)

print("\n--- REKOMENDASI TINDAKAN UNTUK PEMILIK MOBIL ---")
for col, old_v, new_v in zip(actionable_cols, sample_clean[actionable_cols].values[0], opt_vals):
    delta = new_v - old_v
    if abs(delta) < 0.5:
        status = "Pertahankan"
    elif delta < 0:
        status = f"Turunkan {abs(delta):.1f}"
    else:
        status = f"Naikkan {abs(delta):.1f}"
        
    print(f"• {col:<22}: {old_v:6.1f}  ──>  {new_v:6.1f}  [{status}]")

Sampel Mobil Terpilih : Indeks 2542
Probabilitas Awal     : 61.41% (BERBAHAYA)
Probabilitas Baru     : 60.39% (ZONA AMAN)
Waktu Inferensi       : 283.83 ms

--- REKOMENDASI TINDAKAN UNTUK PEMILIK MOBIL ---
• depth_of_discharge    :   66.0  ──>    66.0  [Pertahankan]
• state_of_charge       :   93.0  ──>    63.9  [Turunkan 29.1]
• fast_charge_ratio     :    0.1  ──>     0.1  [Pertahankan]
• hard_braking_score    :   25.6  ──>    25.6  [Pertahankan]
• average_speed         :   19.0  ──>    19.0  [Pertahankan]


In [11]:
# 1. Audit akar masalah mobil indeks 2542
print("--- Diagnostik Internal Mobil Indeks 2542 ---")
diagnostic_cols = [
    'battery_health_percent', 'state_of_health', 'capacity_loss_percent',
    'internal_resistance', 'thermal_runaway_risk', 'thermal_health_score',
    'cell_temperature_max', 'previous_faults', 'vehicle_age_years'
]
print(sample_clean[diagnostic_cols].T)

# 2. Cari kendaraan yang risiko tingginya DIPICU oleh kebiasaan buruk:
# - fast_charge_ratio tinggi (> 0.60)
# - state_of_charge sering penuh (> 90%)
# - hard_braking_score agresif (> 60)
# - tapi kondisi dasar fisik baterai (SOH / battery health) masih cukup baik (> 75%)
habit_abuse_mask = (
    (X['fast_charge_ratio'] >= 0.50) & 
    (X['state_of_charge'] >= 85.0) & 
    (X['hard_braking_score'] >= 50.0) &
    (X['state_of_health'] >= 75.0) &
    (all_preds >= 0.40)
)
abuse_candidates = np.where(habit_abuse_mask)[0]

print(f"\nJumlah mobil berisiko akibat kebiasaan berkendara buruk: {len(abuse_candidates)}")

if len(abuse_candidates) > 0:
    target_idx = abuse_candidates[0]
    sample_habit = X.iloc[[target_idx]].copy()
    init_p = final_model.predict_proba(sample_habit)[0, 1]
    
    print(f"Menguji Sampel Indeks: {target_idx} | Risiko Awal: {init_p * 100:.2f}%")
    
    # Jalankan optimizer pada sampel ini
    opt_v, red_p, exec_ms = robust_tree_counterfactual(sample_habit)
    
    print(f"Probabilitas Baru: {red_p * 100:.2f}% | Selesai dalam: {exec_ms:.2f} ms\n")
    print("--- REKOMENDASI UNTUK PENGEMUDI ---")
    for col, old_v, new_v in zip(actionable_cols, sample_habit[actionable_cols].values[0], opt_v):
        delta = new_v - old_v
        status = "Pertahankan" if abs(delta) < 0.5 else ("Turunkan" if delta < 0 else "Naikkan")
        print(f"• {col:<22}: {old_v:6.1f}  ──>  {new_v:6.1f}  [{status} {abs(delta):.1f}]")
else:
    print("Tidak ditemukan sampel yang pas dengan filter di atas. Kita perlu menerapkan Physics Coupling.")

--- Diagnostik Internal Mobil Indeks 2542 ---
                          2542
battery_health_percent  78.270
state_of_health         78.710
capacity_loss_percent   21.730
internal_resistance      0.481
thermal_runaway_risk    42.560
thermal_health_score    58.030
cell_temperature_max    47.670
previous_faults          2.000
vehicle_age_years       11.200

Jumlah mobil berisiko akibat kebiasaan berkendara buruk: 9
Menguji Sampel Indeks: 1626 | Risiko Awal: 99.23%
Probabilitas Baru: 98.29% | Selesai dalam: 342.13 ms

--- REKOMENDASI UNTUK PENGEMUDI ---
• depth_of_discharge    :   58.9  ──>    85.0  [Naikkan 26.1]
• state_of_charge       :   94.4  ──>    60.0  [Turunkan 34.4]
• fast_charge_ratio     :    0.5  ──>     0.2  [Pertahankan 0.3]
• hard_braking_score    :   78.4  ──>     0.0  [Turunkan 78.4]
• average_speed         :   74.0  ──>    30.0  [Turunkan 44.0]


In [12]:
# 1. Analisis hubungan linear antara perilaku berkendara dengan sensor termal & stres
coupled_targets = ['cell_temperature_max', 'cell_temperature_avg', 'driving_stress_score', 'thermal_runaway_risk']
corrs_behavior = df[actionable_cols + coupled_targets].corr()

print("--- Korelasi Fitur Perilaku terhadap Indikator Termal & Stres ---")
print(corrs_behavior.loc[actionable_cols, coupled_targets])

# 2. Hitung koefisien sensitivitas rata-rata (Delta Slope)
# Seberapa besar penurunan variabel termal per 1 unit penurunan perilaku
slopes = {}
for act in ['fast_charge_ratio', 'hard_braking_score', 'average_speed']:
    slopes[act] = {}
    for tgt in ['cell_temperature_max', 'thermal_runaway_risk', 'driving_stress_score']:
        # Estimasi gradien linear sederhana antar-variabel
        cov = df[[act, tgt]].dropna().cov().iloc[0, 1]
        var = df[act].dropna().var()
        slopes[act][tgt] = cov / var if var != 0 else 0.0

# 3. Fast Tree Counterfactual dengan Physics Coupling
def physics_coupled_counterfactual(car_row, steps=10):
    start_time = time.perf_counter()
    clean_row = car_row.copy()
    
    # Isi NaN pada actionable jika ada
    for col in actionable_cols:
        if pd.isna(clean_row[col].values[0]):
            clean_row[col] = actionable_medians[col]
            
    current_x = clean_row[actionable_cols].values[0].astype(float)
    base_state = clean_row.copy()
    
    best_row = clean_row.copy()
    best_x = current_x.copy()
    best_risk = final_model.predict_proba(best_row)[0, 1]
    
    feature_grids = {
        col: np.linspace(bounds[i][0], bounds[i][1], steps)
        for i, col in enumerate(actionable_cols)
    }
    
    for _ in range(2):
        for i, act_col in enumerate(actionable_cols):
            candidate_vals = feature_grids[act_col]
            batch = pd.concat([best_row] * len(candidate_vals), ignore_index=True)
            batch[act_col] = candidate_vals
            
            # Terapkan efek fisika: Perubahan perilaku menurunkan suhu & risiko termal
            delta_act = candidate_vals - current_x[i]
            if act_col in slopes:
                for tgt, m in slopes[act_col].items():
                    if tgt in batch.columns:
                        # Propagasikan perubahan ke sensor termal
                        batch[tgt] = np.clip(
                            batch[tgt] + (m * delta_act),
                            a_min=df[tgt].min(),
                            a_max=df[tgt].max()
                        )
            
            probs = final_model.predict_proba(batch)[:, 1]
            
            # Cari probabilitas terendah
            min_idx = np.argmin(probs)
            if probs[min_idx] < best_risk:
                best_risk = probs[min_idx]
                best_x[i] = candidate_vals[min_idx]
                best_row = batch.iloc[[min_idx]].copy()

    duration_ms = (time.perf_counter() - start_time) * 1000
    return best_x, best_risk, duration_ms

# 4. Eksekusi Ulang pada Mobil Indeks 1626
print("\n=======================================================")
print(f"Uji Ulang Sampel 1626 dengan Physics Coupling")
print(f"Risiko Awal: {init_p * 100:.2f}%")
print("=======================================================")

coupled_x, coupled_risk, time_ms = physics_coupled_counterfactual(sample_habit)

print(f"Probabilitas Baru (Terkopel): {coupled_risk * 100:.2f}%")
print(f"Waktu Inferensi: {time_ms:.2f} ms\n")

print("--- REKOMENDASI TERVALIDASI FISIKA ---")
for col, old_v, new_v in zip(actionable_cols, sample_habit[actionable_cols].values[0], coupled_x):
    delta = new_v - old_v
    status = "Pertahankan" if abs(delta) < 0.5 else ("Turunkan" if delta < 0 else "Naikkan")
    print(f"• {col:<22}: {old_v:6.1f}  ──>  {new_v:6.1f}  [{status} {abs(delta):.1f}]")

--- Korelasi Fitur Perilaku terhadap Indikator Termal & Stres ---
                    cell_temperature_max  cell_temperature_avg  \
depth_of_discharge             -0.012420             -0.011386   
state_of_charge                 0.003279              0.003332   
fast_charge_ratio              -0.006266             -0.002532   
hard_braking_score              0.001055              0.006090   
average_speed                   0.003623              0.010239   

                    driving_stress_score  thermal_runaway_risk  
depth_of_discharge              0.000378             -0.003992  
state_of_charge                 0.002147              0.011988  
fast_charge_ratio               0.001157              0.002832  
hard_braking_score              0.423697              0.001556  
average_speed                   0.005054              0.006642  

Uji Ulang Sampel 1626 dengan Physics Coupling
Risiko Awal: 99.23%
Probabilitas Baru (Terkopel): 98.35%
Waktu Inferensi: 403.37 ms

--- REKOMENDASI

In [13]:
# 1. Daftar kolom diagnostik internal & vonis akhir yang wajib dibuang
leakage_and_diagnostic_cols = [
    'battery_failure', 'vehicle_id',
    # Degradasi internal fisik
    'battery_health_percent', 'state_of_health', 'capacity_loss_percent',
    'internal_resistance', 'remaining_capacity',
    # Skor komposit turunan
    'thermal_runaway_risk', 'thermal_health_score', 'aging_score',
    'battery_stress_index', 'predicted_remaining_life_cycles',
    # Log kerusakan & anomali historis
    'previous_faults', 'BMS_warning_count', 'sensor_fault_count',
    'abnormal_voltage_events', 'cooling_system_health'
]

# Pastikan hanya menghapus kolom yang benar-benar ada di dataframe
cols_to_drop = [c for c in leakage_and_diagnostic_cols if c in df.columns]

X_ops = df.drop(columns=cols_to_drop).copy()
y = df['battery_failure'].copy()

# 2. Tangani kolom kategorial untuk LightGBM
cat_cols_ops = X_ops.select_dtypes(include=['object', 'string']).columns.tolist()
for col in cat_cols_ops:
    X_ops[col] = X_ops[col].astype('category')

print(f"Jumlah fitur operasional murni: {X_ops.shape[1]} fitur")
print(f"Fitur kategorial: {len(cat_cols_ops)} kolom ({cat_cols_ops})")

# 3. Latih Model Operasional Final
scale_weight = (len(y) - sum(y)) / sum(y)

model_ops = lgb.LGBMClassifier(
    n_estimators=450,
    learning_rate=0.03,
    scale_pos_weight=scale_weight,
    random_state=42,
    importance_type='gain',
    verbose=-1
)
model_ops.fit(X_ops, y)

# Simpan model operasional untuk FastAPI nanti
joblib.dump(model_ops, 'ev_battery_ops_model.joblib')
print("\nModel operasional berhasil dilatih dan disimpan: ev_battery_ops_model.joblib")

# 4. Evaluasi Peringkat Pengaruh Fitur Baru
feat_imp = pd.Series(model_ops.feature_importances_, index=X_ops.columns).sort_values(ascending=False)
print("\n--- Top 15 Fitur Operasional Paling Berpengaruh ---")
print(feat_imp.head(15))

# 5. Uji Responsivitas Simulasi Kebiasaan Pengemudi
actionable_cols = [
    'depth_of_discharge',
    'state_of_charge',
    'fast_charge_ratio',
    'hard_braking_score',
    'average_speed'
]

bounds = [
    (30.0, 75.0),   # batasi depth of discharge agar tidak terlalu tiris
    (50.0, 80.0),   # anjuran batas atas charging harian 80%
    (0.05, 0.30),   # kurangi porsi fast charge
    (0.0, 35.0),    # perhalus gaya pengereman
    (30.0, 80.0)    # kecepatan operasional wajar
]

# Cari sampel mobil yang divonis berisiko tinggi oleh model operasional
ops_preds = model_ops.predict_proba(X_ops)[:, 1]
high_risk_mask = (ops_preds >= 0.70) & (~X_ops[actionable_cols].isnull().any(axis=1))
high_risk_candidates = np.where(high_risk_mask)[0]

target_idx = high_risk_candidates[0]
sample_target = X_ops.iloc[[target_idx]].copy()
initial_risk = model_ops.predict_proba(sample_target)[0, 1]

print(f"\n=======================================================")
print(f"Uji Sampel Indeks {target_idx} | Risiko Awal: {initial_risk * 100:.2f}%")
print(f"=======================================================")

# Fungsi optimasi cepat berbasis batch prediksi
start_t = time.perf_counter()
best_row = sample_target.copy()
best_x = sample_target[actionable_cols].values[0].astype(float)
best_risk = initial_risk

feature_grids = {
    col: np.linspace(bounds[i][0], bounds[i][1], 12)
    for i, col in enumerate(actionable_cols)
}

for _ in range(2):
    for i, col in enumerate(actionable_cols):
        candidate_vals = feature_grids[col]
        batch = pd.concat([best_row] * len(candidate_vals), ignore_index=True)
        batch[col] = candidate_vals
        
        probs = model_ops.predict_proba(batch)[:, 1]
        min_idx = np.argmin(probs)
        
        if probs[min_idx] < best_risk:
            best_risk = probs[min_idx]
            best_x[i] = candidate_vals[min_idx]
            best_row[col] = candidate_vals[min_idx]

exec_ms = (time.perf_counter() - start_t) * 1000

print(f"Waktu Inferensi: {exec_ms:.2f} ms")
print(f"Risiko Baru Setelah Kebiasaan Diubah: {best_risk * 100:.2f}%\n")

print("--- REKOMENDASI PERUBAHAN UNTUK PENGEMUDI ---")
for col, old_v, new_v in zip(actionable_cols, sample_target[actionable_cols].values[0], best_x):
    delta = new_v - old_v
    status = "Pertahankan" if abs(delta) < 0.5 else ("Turunkan" if delta < 0 else "Naikkan")
    print(f"• {col:<22}: {old_v:6.1f}  ──>  {new_v:6.1f}  [{status} {abs(delta):.1f}]")

Jumlah fitur operasional murni: 53 fitur
Fitur kategorial: 9 kolom (['vehicle_brand', 'vehicle_model', 'vehicle_type', 'battery_manufacturer', 'battery_chemistry', 'drive_type', 'fleet_or_private', 'battery_serial', 'terrain_type'])

Model operasional berhasil dilatih dan disimpan: ev_battery_ops_model.joblib

--- Top 15 Fitur Operasional Paling Berpengaruh ---
cell_temperature_max    164694.686899
charge_efficiency       108476.395025
vehicle_model            72740.798697
discharge_efficiency     55113.112734
vehicle_age_years        47668.672782
voltage_imbalance        30448.818006
cell_temperature_avg     20958.346056
maintenance_score        20739.909138
cell_voltage_std         17560.857353
dust_exposure            10659.533891
last_service_days        10493.034914
depth_of_discharge        7783.551219
home_charging_ratio       6548.364426
odometer_km               6194.620626
pack_voltage              5964.391577
dtype: float64

Uji Sampel Indeks 2 | Risiko Awal: 80.29%
Waktu In

In [14]:
import json
import joblib

# 1. Drop battery_serial jika ada, lalu fit ulang secara cepat
if 'battery_serial' in X_ops.columns:
    X_ops = X_ops.drop(columns=['battery_serial'])
    print("Kolom battery_serial berhasil dibuang.")

# Fit ulang model bersih
model_ops.fit(X_ops, y)
joblib.dump(model_ops, 'ev_battery_ops_model.joblib')

# 2. Siapkan metadata skema untuk FastAPI
cat_columns = X_ops.select_dtypes(include=['category']).columns.tolist()
num_columns = X_ops.select_dtypes(exclude=['category']).columns.tolist()

metadata = {
    "all_features": list(X_ops.columns),
    "categorical_features": cat_columns,
    "numeric_features": num_columns,
    "actionable_features": actionable_cols,
    "bounds": {col: bounds[i] for i, col in enumerate(actionable_cols)},
    "medians": X_ops[num_columns].median().to_dict(),
    "categorical_defaults": {col: str(X_ops[col].mode()[0]) for col in cat_columns}
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Berhasil mengekspor:")
print("1. ev_battery_ops_model.joblib (Model)")
print("2. model_metadata.json (Skema & Parameter)")

Kolom battery_serial berhasil dibuang.
Berhasil mengekspor:
1. ev_battery_ops_model.joblib (Model)
2. model_metadata.json (Skema & Parameter)
